# Mercari(일본 중고거래 플랫폼) 가격 예측 프로젝트

## 프로젝트 흐름  
1. 환경 설정 및 데이터 로드
2. 데이터 전처리 (결측치, 카테고리 분리)
3. 텍스트 특징 추출 (NLP_자연어 처리)
4. 새로운 피처 생성
5. 모델 학습
6. 가격 예측 & 제출

### 1. 환경설정 및 데이터 로드

In [34]:
# 1. 환경설정
pip install pandas numpy scikit-learn lightgbm nltk scipy

Note: you may need to restart the kernel to use updated packages.


In [35]:
# 1. 환경설정
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# 1. 데이터 로드
train = pd.read_csv('train.tsv', sep='\t')
test  = pd.read_csv('test.tsv',  sep='\t')

print(train.shape)
print(train.head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\berga\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\berga\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\berga\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


(1482535, 8)
   train_id                                 name  item_condition_id  \
0         0  MLB Cincinnati Reds T Shirt Size XL                  3   
1         1     Razer BlackWidow Chroma Keyboard                  3   
2         2                       AVA-VIV Blouse                  1   
3         3                Leather Horse Statues                  1   
4         4                 24K GOLD plated rose                  1   

                                       category_name brand_name  price  \
0                                  Men/Tops/T-shirts        NaN   10.0   
1  Electronics/Computers & Tablets/Components & P...      Razer   52.0   
2                        Women/Tops & Blouses/Blouse     Target   10.0   
3                 Home/Home Décor/Home Décor Accents        NaN   35.0   
4                            Women/Jewelry/Necklaces        NaN   44.0   

   shipping                                   item_description  
0         1                                 No des

In [36]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1482535 entries, 0 to 1482534
Data columns (total 8 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   train_id           1482535 non-null  int64  
 1   name               1482535 non-null  object 
 2   item_condition_id  1482535 non-null  int64  
 3   category_name      1476208 non-null  object 
 4   brand_name         849853 non-null   object 
 5   price              1482535 non-null  float64
 6   shipping           1482535 non-null  int64  
 7   item_description   1482529 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 90.5+ MB


In [37]:
train.describe()

,train_id,item_condition_id,price,shipping
count,1.482535e+06,1.482535e+06,1.482535e+06,1.482535e+06
mean,7.412670e+05,1.907380e+00,2.673752e+01,4.472744e-01
std,4.279711e+05,9.031586e-01,3.858607e+01,4.972124e-01
min,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
25%,3.706335e+05,1.000000e+00,1.000000e+01,0.000000e+00
50%,7.412670e+05,2.000000e+00,1.700000e+01,0.000000e+00
75%,1.111900e+06,3.000000e+00,2.900000e+01,1.000000e+00
max,1.482534e+06,5.000000e+00,2.009000e+03,1.000000e+00


In [90]:
train.isna().sum()

train_id                   0
name                       0
item_condition_id          0
category_name           6327
brand_name                 0
price                      0
shipping                   0
item_description           6
cat_1                      0
cat_2                      0
cat_3                      0
cat_4                1478146
cat_5                1479476
clean_desc                 0
clean_name                 0
text_combined              0
color                      0
material                   0
is_bundle                  0
cat_1_enc                  0
cat_2_enc                  0
cat_3_enc                  0
brand_name_enc             0
color_enc                  0
material_enc               0
dtype: int64

In [38]:
train.isna().mean()

train_id             0.000000
name                 0.000000
item_condition_id    0.000000
category_name        0.004268
brand_name           0.426757
price                0.000000
shipping             0.000000
item_description     0.000004
dtype: float64

In [39]:
train['item_condition_id'].unique()

array([3, 1, 2, 4, 5])

In [40]:
train['shipping'].unique()

array([1, 0])

### 2. category_name 분리 ('/' 구분) 및 데이터 전처리  

#### 2-1.category_name 분리

In [41]:
train['category_name'].nunique()

1287

In [42]:
train['category_name'].unique()

array(['Men/Tops/T-shirts',
       'Electronics/Computers & Tablets/Components & Parts',
       'Women/Tops & Blouses/Blouse', ..., 'Handmade/Jewelry/Clothing',
       'Vintage & Collectibles/Supplies/Ephemera',
       'Handmade/Pets/Blanket'], shape=(1288,), dtype=object)

In [64]:
def split_category(category):
    """카테고리를 최대 3단계로 분리"""
    if pd.isnull(category):
        return pd.Series(['Unknown', 'Unknown', 'Unknown'])
    parts = category.split('/', 2)  # 최대 3개로 분리
    while len(parts) < 3:
        parts.append('Unknown')
    return pd.Series(parts[:3])

# 카테고리 분리 적용
train[['cat_1', 'cat_2', 'cat_3']] = train['category_name'].apply(split_category)
test[['cat_1',  'cat_2', 'cat_3']] = test['category_name'].apply(split_category)

print(train['cat_1'].value_counts().head(10))  # 상위 카테고리 확인

cat_1
Women                     664385
Beauty                    207828
Kids                      171689
Electronics               122690
Men                        93680
Home                       67871
Vintage & Collectibles     46530
Other                      45351
Handmade                   30842
Sports & Outdoors          25342
Name: count, dtype: int64


In [65]:
train.head(5)

,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description,cat_1,cat_2,cat_3,cat_4,cat_5,clean_desc,clean_name,text_combined,color,material,is_bundle
0,0,MLB Cincinnati Reds T Shirt Size XL,3,Men/Tops/T-shirts,Unknown,10.0,1,No description yet,Men,Tops,T-shirts,None,None,description yet,mlb cincinnati reds shirt size xl,mlb cincinnati reds shirt size xl description yet,red,unknown,0
1,1,Razer BlackWidow Chroma Keyboard,3,Electronics/Computers & Tablets/Components & P...,Razer,52.0,0,This keyboard is in great condition and works ...,Electronics,Computers & Tablets,Components & Parts,None,None,keyboard condition works like came box ports t...,razer blackwidow chroma keyboard,razer blackwidow chroma keyboard keyboard cond...,black,unknown,0
2,2,AVA-VIV Blouse,1,Women/Tops & Blouses/Blouse,Target,10.0,1,Adorable top with a hint of lace and a key hol...,Women,Tops & Blouses,Blouse,None,None,adorable top hint lace key hole back pale pink...,ava viv blouse,ava viv blouse adorable top hint lace key hole...,white,unknown,0
3,3,Leather Horse Statues,1,Home/Home Décor/Home Décor Accents,Unknown,35.0,1,New with tags. Leather horses. Retail for [rm]...,Home,Home Décor,Home Décor Accents,None,None,tags leather horses retail rm stand foot high ...,leather horse statues,leather horse statues tags leather horses reta...,unknown,leather,0
4,4,24K GOLD plated rose,1,Women/Jewelry/Necklaces,Unknown,44.0,0,Complete with certificate of authenticity,Women,Jewelry,Necklaces,None,None,complete certificate authenticity,24k gold plated rose,24k gold plated rose complete certificate auth...,gold,unknown,0


In [66]:
# 중분류 카테고리 확인
train['cat_2'].unique()

array(['Tops', 'Computers & Tablets', 'Tops & Blouses', 'Home Décor',
       'Jewelry', 'Other', 'Swimwear', 'Apparel', 'Collectibles',
       'Makeup', 'Fragrance', 'Dresses', 'Office supplies', 'Shoes',
       'Gear', 'Athletic Apparel', 'Cell Phones & Accessories', 'Jeans',
       'Underwear', 'Skin Care', 'Toys', "Women's Handbags",
       'Video Games & Consoles', 'Coats & Jackets', 'Pants', 'Girls (4+)',
       'Antique', 'Kitchen & Dining', 'Sweaters', 'Boys 0-24 Mos',
       'Girls 0-24 Mos', 'Maternity', 'Bedding', 'Exercise',
       'Trading Cards', 'Boys (4+)', 'Storage & Organization', 'Fan Shop',
       'Girls 2T-5T', "Men's Accessories", 'Boys 2T-5T',
       "Women's Accessories", 'Daily & Travel items', 'Unknown', 'Skirts',
       'Hair Care', 'Pet Supplies', 'Book', 'Tools & Accessories',
       'Team Sports', 'Home Appliances', 'Accessories', 'Bags and Purses',
       'Sweats & Hoodies', 'Shorts', 'TV, Audio & Surveillance',
       'Outdoors', 'Bath & Body', 'Car Seats

In [67]:
train['cat_2'].nunique()

114

In [68]:
train['cat_3'].unique()

array(['T-shirts', 'Components & Parts', 'Blouse', 'Home Décor Accents',
       'Necklaces', 'Other', 'Two-Piece', 'Girls', 'Doll', 'Face',
       'Women', 'Above Knee, Mini', 'School Supplies', 'Boots',
       'Makeup Sets', 'Eyes', 'Backpacks & Carriers', 'Makeup Palettes',
       'Tank, Cami', 'Sports Bras', 'Cell Phones & Smartphones',
       'Chargers & Cradles', 'T-Shirts', 'Athletic',
       'Cases, Covers & Skins', 'Pants, Tights, Leggings', 'One-Piece',
       'Boot Cut', 'Bras', 'Stuffed Animals & Plush', 'Totes & Shoppers',
       'Shirts & Tops', 'Consoles', 'Glass', 'Vest', 'Arts & Crafts',
       'Capris, Cropped', 'Messenger & Crossbody', 'Shoes',
       'Collectibles', 'Coffee & Tea Accessories', 'Brooch', 'Headsets',
       'Rings', 'Shorts', 'Fleece Jacket', 'Dolls & Accessories',
       'Crewneck', 'Jackets', 'Home Fragrance', 'Accessories',
       'Tops & Blouses', 'Sheets & Pillowcases', 'Fitness technology',
       'Dress Up & Pretend Play', 'Animation',
       'J

In [70]:
train['cat_3'].nunique()

872

#### 2-2. 'brand_name' column 확인 및 결측치 처리 : 4,809개 _ 유명브랜드와 그 외 브랜드 구분

In [71]:
train['brand_name'].unique()

array(['Unknown', 'Razer', 'Target', ..., 'Astroglide', 'Cumberland Bay',
       'Kids Only'], shape=(4813,), dtype=object)

In [72]:
train['brand_name'].nunique()

4813

In [81]:
print(train['brand_name'].value_counts().head(25))  # 전체 상위 브랜드 확인

brand_name
Unknown              597648
Nike                  59443
PINK                  54088
Victoria's Secret     48036
LuLaRoe               31024
Apple                 24979
Nintendo              16478
Michael Kors          15575
Lululemon             15228
FOREVER 21            15186
American Eagle        13254
Coach                 12641
Adidas                12386
Rae Dunn              12305
Sephora               12172
Disney                10360
Bath & Body Works     10354
Funko                  9237
Under Armour           8461
Sony                   8231
Samsung                7900
Old Navy               7567
Hollister              6948
Carter's               6385
Urban Decay            6210
Name: count, dtype: int64


In [60]:
# 유명 브랜드 리스트 (필요시 확장)
KNOWN_BRANDS = [
    'Nike', 'Adidas', 'Apple', 'Samsung', 'Zara', 'H&M',
    'Victoria Secret', 'Louis Vuitton', 'Gucci', 'Prada',
    'Lululemon', 'Forever 21', 'Target', 'Nintendo', 'Sony',
    'Michael Kors', 'Coach', 'Tory Burch', 'Ralph Lauren'
]

def extract_brand(row):
    """name, category, description에서 브랜드 추출 시도"""
    if pd.notnull(row['brand_name']):
        return row['brand_name']
    
    text = ' '.join([
        str(row['name']),
        str(row['item_description'])
    ]).lower()
    
    for brand in KNOWN_BRANDS:
        if brand.lower() in text:
            return brand
    
    return 'Unknown'

train['brand_name'] = train.apply(extract_brand, axis=1)
test['brand_name']  = test.apply(extract_brand,  axis=1)

### 3. 텍스트 핵심 키워드 추출 (자연어 정리) : TF-IDF 방식 사용

In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# 불용어 설정 (영어 기본 + 커스텀 추가)
stop_words = set(stopwords.words('english'))
custom_stops = {'item', 'please', 'great', 'good', 'nice', 'make', 'used', 'new'}
stop_words.update(custom_stops)

def clean_text(text):
    """텍스트 정제: 소문자화, 특수문자 제거, 불용어 제거"""
    if pd.isnull(text):
        return 'unknown'
    
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # 특수문자 제거
    
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    
    return ' '.join(tokens)

# 텍스트 정제 적용
train['clean_desc'] = train['item_description'].apply(clean_text)
train['clean_name'] = train['name'].apply(clean_text)
test['clean_desc']  = test['item_description'].apply(clean_text)
test['clean_name']  = test['name'].apply(clean_text)

# name + description 합치기
train['text_combined'] = train['clean_name'] + ' ' + train['clean_desc']
test['text_combined']  = test['clean_name']  + ' ' + test['clean_desc']

# TF-IDF 벡터화 (상위 50,000 단어만 사용)
tfidf_desc = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))

X_train_text = tfidf_desc.fit_transform(train['text_combined'])
X_test_text  = tfidf_desc.transform(test['text_combined'])

print(f"TF-IDF 행렬 크기: {X_train_text.shape}")

TF-IDF 행렬 크기: (1482535, 50000)


### 4. 제품 특징 feature 생성 : 색깔, 소재

In [62]:
# 색상 키워드
COLORS = ['red','blue','green','black','white','pink','purple',
          'yellow','orange','grey','gray','brown','gold','silver']

# 소재 키워드
MATERIALS = ['cotton','leather','silk','wool','polyester','nylon',
             'denim','linen','velvet','suede','rubber','metal']

def extract_features(text):
    """텍스트에서 색상, 소재 추출"""
    text = str(text).lower()
    colors    = [c for c in COLORS    if c in text]
    materials = [m for m in MATERIALS if m in text]
    return (
        colors[0]    if colors    else 'unknown',
        materials[0] if materials else 'unknown'
    )

train[['color', 'material']] = train['text_combined'].apply(
    lambda x: pd.Series(extract_features(x))
)
test[['color', 'material']] = test['text_combined'].apply(
    lambda x: pd.Series(extract_features(x))
)

# 묶음/개수 추출 (예: "set of 3", "bundle", "lot of")
def has_bundle(text):
    text = str(text).lower()
    return int(any(w in text for w in ['bundle', 'set of', 'lot of', 'pack of', 'set']))

train['is_bundle'] = train['text_combined'].apply(has_bundle)
test['is_bundle']  = test['text_combined'].apply(has_bundle)

### 5-6. 모델 학습 및 가격 예측

In [73]:
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

# 레이블 인코딩 (카테고리형 변수)
cat_cols = ['cat_1', 'cat_2', 'cat_3', 'brand_name',
            'color', 'material']

le = LabelEncoder()
for col in cat_cols:
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col + '_enc'] = le.transform(train[col].astype(str))
    test[col  + '_enc'] = le.transform(test[col].astype(str))

# 수치형 피처 선택
num_features = ['item_condition_id', 'shipping', 'is_bundle',
                'cat_1_enc', 'cat_2_enc', 'cat_3_enc',
                'brand_name_enc', 'color_enc', 'material_enc']

X_train_num = csr_matrix(train[num_features].values)
X_test_num  = csr_matrix(test[num_features].values)

# TF-IDF + 수치형 합치기
X_train_all = hstack([X_train_text, X_train_num])
X_test_all  = hstack([X_test_text,  X_test_num])

# 타겟: 가격 로그 변환 (왜도 줄이기)
y_train = np.log1p(train['price'])

# 학습/검증 분리
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_all, y_train, test_size=0.1, random_state=42
)

# LightGBM 모델 학습
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.1,
    'num_leaves': 127,
    'n_estimators': 1000,
    'min_child_samples': 20,
    'random_state': 42
}

model = lgb.LGBMRegressor(**params)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

# 예측
pred_log = model.predict(X_test_all)
pred_price = np.expm1(pred_log)  # 로그 역변환
pred_price = np.clip(pred_price, 0, None)  # 음수 방지

# 제출 파일 생성
submission = pd.read_csv('sample_submission.csv')
submission['price'] = pred_price
submission.to_csv('my_submission.csv', index=False)
print("제출 파일 생성 완료!")
print(submission.head())

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 110.524020 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1123105
[LightGBM] [Info] Number of data points in the train set: 1334281, number of used features: 50006
[LightGBM] [Info] Start training from score 2.978544
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.507799
[200]	valid_0's rmse: 0.486044
[300]	valid_0's rmse: 0.476043
[400]	valid_0's rmse: 0.469779
[500]	valid_0's rmse: 0.465474
[600]	valid_0's rmse: 0.462057
[700]	valid_0's rmse: 0.459193
[800]	valid_0's rmse: 0.457182
[900]	valid_0's rmse: 0.455465
[1000]	valid_0's rmse: 0.453972
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 0.453972


c:\Users\berga\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


제출 파일 생성 완료!
   test_id      price
0        0   7.548861
1        1  11.036666
2        2  33.649194
3        3  13.766893
4        4   6.926652
